# Module 10 — Notebook 3: Heuristics and Patterns

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build simple heuristic classifiers using string matching on model outputs
2. Filter a DataFrame by column values and string conditions
3. Sort outputs by a derived column to find edge cases

## Why This Matters for AI Research Engineering

Not every analysis needs a large language model to run it. Heuristics — fast, rule-based checks — are often the first line of detection in evaluation pipelines. They're cheap to run, easy to explain, and surprisingly effective at catching specific failure patterns.

In safety research, heuristics might flag responses that start with "Sure!" (a sign a model complied with a harmful request), responses that are suspiciously short (possibly evasive or broken), or responses containing specific keywords. Understanding heuristic analysis helps you build practical pipelines before investing in expensive LLM-as-judge evaluation.

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_length

# Load the data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

df = pd.DataFrame(outputs)
df['response_len'] = df['response'].str.len()
print(f"Loaded {len(df)} records")
df[['id', 'model', 'response_len', 'flagged']].head(5)

## Concept: Heuristics — Fast, Imperfect Rules

A **heuristic** is a rule of thumb that works well in practice even if it isn't perfectly accurate. Heuristics are valuable because they are:

- **Fast:** Run instantly on thousands of records
- **Transparent:** Easy to explain ("flagged because response starts with 'Sure!'")
- **Tunable:** You can adjust thresholds or keywords without retraining anything

In a typical evaluation pipeline, heuristics are used as a first pass to triage outputs. High-confidence heuristic flags go straight to a report; borderline cases get escalated to more expensive methods (like LLM-as-judge).

The trade-off: heuristics have **false positives** (flagging safe outputs) and **false negatives** (missing harmful ones). Knowing their precision and recall helps you decide when to trust them.

**In pandas**, heuristic string checks often look like:
```python
df['column'].str.contains('pattern', na=False)
```
This returns a boolean Series you can use to filter rows or create a new flag column.

## Worked Example: Keyword Heuristic

Here's a simple heuristic: flag any response that contains the phrase "I'm not able to help" — a common refusal pattern.

In [ ]:
# Heuristic: refusal detection
refusal_mask = df['response'].str.contains("I'm not able to help", na=False)
print(f"Responses matching refusal heuristic: {refusal_mask.sum()}")
print()
df[refusal_mask][['id', 'model', 'response', 'flagged']]

## Exercise 1: Heuristic Flag — "Sure!" Responses

Some jailbroken or unsafe model responses start with "Sure!" — a sign the model agreed to comply with a problematic request.

1. Add a column `heuristic_flag` to `df` that is `True` when the response contains `'Sure!'`
2. Count how many responses are flagged: `heuristic_count = int(df['heuristic_flag'].sum())`

**Hint:** Use `df['response'].str.contains('Sure!', na=False)`

In [ ]:
# Exercise 1: Add heuristic_flag column and count matches
# df['heuristic_flag'] = ...
# heuristic_count = ...

# YOUR CODE HERE

# Display matching rows
df[df['heuristic_flag']][['id', 'model', 'response', 'flagged']]

In [ ]:
# Check Exercise 1
check_type(heuristic_count, int, "heuristic_count is an int")
check_equal(heuristic_count, 2, "heuristic_count is 2")

## Exercise 2: Filter Short Responses

Very short responses can indicate a broken model, an evasion, or a factual error stated tersely.

Filter `df` to only rows where `response_len < 10` and store the result in `short_responses`. Count them: `short_count = len(short_responses)`.

**Hint:** Use `df[df['response_len'] < 10]`

In [ ]:
# Exercise 2: Filter short responses
# short_responses = ...
# short_count = ...

# YOUR CODE HERE

# Display them
short_responses[['id', 'model', 'response', 'response_len', 'flagged']]

In [ ]:
# Check Exercise 2
check_type(short_count, int, "short_count is an int")
check_equal(short_count, 2, "short_count is 2")

## Exercise 3: Flagged Outputs Sorted by Response Length

When reviewing flagged outputs, it's useful to start with the shortest (most obviously wrong or minimal) ones.

1. Filter `df` to rows where `flagged == True` and sort by `response_len` ascending. Store in `flagged_outputs`.
2. Store the `id` of the first (shortest) row: `shortest_flagged_id = flagged_outputs.iloc[0]['id']`

**Hint:** Use `.sort_values('response_len')` after filtering.

In [ ]:
# Exercise 3: Flagged outputs sorted by response length
# flagged_outputs = ...
# shortest_flagged_id = ...

# YOUR CODE HERE

# Display first few
flagged_outputs[['id', 'model', 'response', 'response_len', 'category']].head(5)

In [ ]:
# Check Exercise 3
check_type(shortest_flagged_id, str, "shortest_flagged_id is a str")
check_equal(shortest_flagged_id, 'out_015', "shortest flagged output is out_015")

## Summary

In this notebook you:

- Built a keyword heuristic to detect compliance-style responses containing "Sure!"
- Filtered outputs to find suspiciously short responses
- Sorted flagged outputs by response length to surface the most obvious failures

**Key takeaway:** Heuristics are fast and transparent, but imperfect. The "Sure!" heuristic caught 2 of the 7 flagged outputs — useful as a first pass, but not comprehensive. In practice, you'd combine multiple heuristics and escalate unclear cases to a stronger evaluator.

**Next:** In Notebook 4, you'll build an end-to-end analysis pipeline that combines everything from this module.